# SNR Optimization Pipeline

This pipeline contains the initial steps of the Py4DEEG approach for performing system identification of cortical circuits. 

1. Loads & preprocesses EEG recordings.
2. Obtains the event related potential (ERP).
3. Performs source localization.
4. Performs an SNR assessment in the electrode and source spaces.

The folder with all the processed data is about 65 GB.

To run this code, one must setup an environment for the jupyter notebook that is mne based.
Follow the instructions on the MNE website: https://mne.tools/stable/install/manual_install.html#manual-install

## Imports

In [1]:
# -*- coding: utf-8 -*-

import h5py
import matplotlib
matplotlib.use('Qt5Agg') # makes plots appear in new window
import matplotlib.pyplot as plt
import sys
import os
import pandas  as pd
import seaborn as sns

sys.path.append("functions_01") 
from setparm               import set_parm
from eegrelatedfuncs       import eeg_preprocess
from convertfuncs          import convert_to_erp
from headtemplatefuncs     import load_head
from leadfieldfuncs        import prepare_source_space, prepare_leadfield
from srclocalizationfuncs  import source_localization, plot_stc, plot_same_source_different_methods,       \
                                  source_localization_split, plot_nmse_across_dists,                       \
                                  plot_split_nmse_participants
from snrfuncs              import measure_snr_el, measure_snr_src_distspaces,  measure_snr_src_repnum,     \
                                  measure_snr_src_multi_subject, plot_snr_el_multi_subject,                \
                                  load_all_snr_data, fit_lme_snr_model, measure_mse_between_splits,        \
                                  measure_snr_src_repnum_dist,  plot_combined_snr_hemispheric              

## Prepare subject path

In [2]:
subject  = '01'   
protocol = 'P0'
session  = 'S1'
cwd      = os.getcwd()
subjpath = os.path.join(cwd,'data','raw_data',protocol,subject)
parm     = set_parm(subject, session, protocol)

Setting up parameters: started
Setting up parameters: completed


## Preprocess Raw EEG

In this section the raw EEG data are pre-processed. The code is included in this package for academic clarity, but for the sake of low memory consumption, the already processed files are included online.

In [7]:
eeg_preprocess(parm, False,False,False)  # (preica code, ica code, postica code)

## Convert to ERP

In this section, the ERP is measured for the subject 'subject'. This part generates the first figure pf the manuscript. Subject 01 was used for the figure.

In [8]:
convert_to_erp(parm)

Converting epochs to erp: started
Reading C:\Users\ioanniskyriazi\3_MainPython\Py4DEEG\data\processed_data\P0\06\S1\eeg\subject-epo.fif ...
    Found the data of interest:
        t =    -100.10 ...     200.20 ms
        0 CTF compensation matrices available
Not setting metadata
2000 matching events found
No baseline correction applied
0 projection items activated
Converting epochs to erp: completed


## Load Head & Brain

This section loads and prepares the head and brain files. No need to run it, the files are already in the processed data folders.

In [4]:
load_head(parm)

## Prepare source space

This section creates the 4 source spaces (5 mm, 10 mm, 15 mm and 20 mm). No need to run it, results are in processed_data\P0\{subject}\S1\leadfield.

In [12]:
prepare_source_space(parm)

## Prepare leadfield

In [ ]:
prepare_leadfield(parm)

## Perform Source Localization

Performs source localization for all participants, on all methods and all source spaces.

In [10]:
source_localization(parm)         # One group of 2000 epochs 

In [ ]:
source_localization_split(parm)   # Two groups of 1000 epochs each 

## Compare Source Localization Methods

Finds the source with the highest SNR for each participant and plots group total power and baseline per method and number of trials.

In [15]:
plot_same_source_different_methods(parm) 

## Assess SNR in Sensor Space

For all participants creates the Monte Carlo curves for electrodes CP3, CPz and CP4. Essentially produces Figure 2 of the manuscript, as well as Supplementary Figure 1. 

In [17]:
# File definition
csv_file_el = f"snr_{subject}_el.csv"  
measure_snr_el(parm,  output_csv=csv_file_el)  # ALREADY CALCULATED - NO NEED TO RUN AGAIN

In [3]:
plot_snr_el_multi_subject(parm)  # When all snr_{subject}_el.csv files have been generated, execute this. 

Plotting SNR, signal power, and noise power: started

SNR SUMMARY TABLE (Mean ± SD) - By Number of Trials
Trials      CP3 (Mean±SD)       Gain      Efficiency  CPz (Mean±SD)       Gain      Efficiency  CP4 (Mean±SD)       Gain      Efficiency  
------------------------------------------------------------------------------------------------------------------------
125         5.70±2.52                                 2.75±1.45                                 1.68±0.48                                 
250         8.65±4.16           +2.95     0.024       3.81±2.21           +1.06     0.008       1.87±0.68           +0.19     0.002       
500         11.99±6.53          +3.34     0.013       4.57±2.65           +0.76     0.003       2.12±0.83           +0.25     0.001        ← LOW-EFF
1000        15.13±7.98          +3.14     0.006       5.50±3.18           +0.93     0.002       2.81±1.30           +0.69     0.001        ← LOW-EFF
1500        18.64±10.47         +3.51     0.007       6.40

## Assess SNR in Source Space

In [27]:
# File definition
csv_file_src_dist   = f"snr_{subject}_src_dist.csv"
csv_file_src        = f"snr_{subject}_src.csv"


method = 'eLORETA'  #Choose: sLORETA, eLORETA, MNE, dSPM

#measure_snr_src_repnum(parm, method, csv_file_src) 
#measure_snr_src_distspaces(parm,method, csv_file_src_dist)
#measure_snr_src_repnum_dist(parm, method,dist='5', output_csv ='snr_dSPM_5mm_source.csv')

measure_snr_src_multi_subject(parm, method,filter_s1_only=True)  #USED

## Compare Sensor & Source Spaces

When executed, it produces panels of C and D of Figure 3 of the manuscript.

In [30]:
plot_combined_snr_hemispheric(parm, method='eLORETA')

## Compare MSE between the 2 Splits

When exectued it produces Figure 4 of the Supplementary Material of the manuscript.

In [ ]:
measure_mse_between_splits(parm)  # Splits (2 groups of 1000 trials each)

## Linear Mixed Effects Model

In [ ]:
# Define parameters
methods = ['eLORETA', 'sLORETA', 'MNE', 'dSPM']
dists   = [5, 10, 15, 20]

snr_df  = load_all_snr_data(parm, methods, dists)
result  = fit_lme_snr_model(snr_df)   
result